# Carga histórica via Google Earth Engine

Decisão registrada na issue de avaliação Databricks vs Earth Engine
(Databricks descartado por restrição de rede no plano gratuito).

**Modelo:** ACCESS-CM2 | **Cenário:** historical | **Período:** 1985–2014 (30 anos)


In [1]:
import os
from dotenv import load_dotenv
import ee

load_dotenv()

ee.Authenticate()
ee.Initialize(project=os.environ["EE_PROJECT_ID"])

c:\Users\felfe\miniconda3\envs\projeto_fase1\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## Coleção e recorte pro Brasil


In [2]:
brasil = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq("country_na", "Brazil"))

colecao = (
    ee.ImageCollection("NASA/GDDP-CMIP6")
    .filter(ee.Filter.eq("model", "ACCESS-CM2"))
    .filter(ee.Filter.eq("scenario", "historical"))
    .filterDate("1985-01-01", "2014-12-31")
    .select("pr")
)

print(colecao.size().getInfo())  # deve retornar 10950 (30 anos x 365)

10956


#### Agora vou fazer os codigos de teste mas devo rodar so amanha de manha (esse markdown vai sumir)


---

#### calcular a média diária sobre o Brasil pra cada imagem, direto no servidor


In [3]:
def calcular_media_diaria(imagem):
    media = imagem.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=brasil.geometry(),
        scale=25000,  # 0.25 grau em metros
        maxPixels=1e9
    )
    return ee.Feature(None, {
        "data": imagem.date().format("YYYY-MM-dd"),
        "valor": media.get("pr")
    })

serie = colecao.map(calcular_media_diaria)

## Exportar como CSV para o `Google Drive`

Com a assinatura do google pro, tenho 400gb de armazenamento na nuvem, o que facilita e não tras gargalo pra minha maquina.


In [4]:
task = ee.batch.Export.table.toDrive(
    collection=serie,
    description="pr_historico_1985_2014",
    fileFormat="CSV"
)
task.start()

In [5]:
# Pra acompanhar o progresso
task.status()

{'state': 'READY',
 'description': 'pr_historico_1985_2014',
 'priority': 100,
 'creation_timestamp_ms': 1789756471460,
 'update_timestamp_ms': 1789756471460,
 'start_timestamp_ms': 0,
 'task_type': 'EXPORT_FEATURES',
 'id': 'MR7DFOLGB774CTP4WH5ECFZG',
 'name': 'projects/project-dbd596b0-fffb-4e15-b5f/operations/MR7DFOLGB774CTP4WH5ECFZG'}